# Notebook 1 - Dataset Understanding
Sprint 4 - EDA

In this notebook I load my dataset and try to understand it before doing any analysis. I cover what a dataset is, features, target, variable types, identifier columns, metadata and dataset dimensions.

## Dataset I selected

- **Name:** Telco Customer Churn
- **Source:** Kaggle (IBM sample dataset), file `WA_Fn-UseC_-Telco-Customer-Churn.csv`
- **Domain:** Telecom
- **Rows:** 7043
- **Columns:** 21
- **Target:** `Churn` (Yes/No), so it is a classification problem

**Important columns**
- `customerID` - unique id of each customer
- `tenure` - how many months the customer has been with the company
- `Contract` - Month-to-month, One year or Two year
- `InternetService` - DSL, Fiber optic or No
- `PaymentMethod` - how the customer pays the bill
- `MonthlyCharges` - amount charged every month
- `TotalCharges` - total amount charged so far
- `Churn` - whether the customer left the company

**Why I chose it:** It has both numerical and categorical columns, a clear target, and some data quality problems (like TotalCharges being stored as text), which will be useful in the next notebooks. Customer churn is also a real business problem, because keeping a customer is cheaper than finding a new one.

### Why this dataset?
- **Multiple features** (20 columns besides the ID and target) from customer, service and billing perspectives.
- Has **numerical**  and **categorical** columns (contract, payment method, services, etc.).
- Has **realistic data-quality issues**: TotalCharges is stored as text and contains blank values, and SeniorCitizen is numeric even though it is really a category. These are good material for the Data Quality notebook.
- Strong **business relevance**: customer churn directly affects revenue, and acquiring a new customer usually costs more than retaining one.
- Has a clear **target variable**, so every notebook in this sprint (including Target Analysis) is applicable.

## Loading the data

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


read_csv loads the file into a DataFrame and head() shows the first 5 rows. Each row is one customer and each column is one detail about that customer.

## 1. What is a Dataset?

A dataset is a collection of data arranged in a table. Rows are the records (observations) and columns are the variables that describe each record.

**Example:** A student table with columns Name, Age, Marks. One row per student.  
**In my dataset:** One row per telecom customer, with columns for their plan, charges and whether they left.  
**Why it matters:** We must know what one row stands for. Here, one row = one customer, so the model will make one prediction per customer.

In [4]:
print(type(df))
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# one single record
df.iloc[0]

<class 'pandas.DataFrame'>
Rows: 7043
Columns: 21


customerID                7590-VHVEG
gender                        Female
SeniorCitizen                      0
Partner                          Yes
Dependents                        No
tenure                             1
PhoneService                      No
MultipleLines       No phone service
InternetService                  DSL
OnlineSecurity                    No
OnlineBackup                     Yes
DeviceProtection                  No
TechSupport                       No
StreamingTV                       No
StreamingMovies                   No
Contract              Month-to-month
PaperlessBilling                 Yes
PaymentMethod       Electronic check
MonthlyCharges                 29.85
TotalCharges                   29.85
Churn                             No
Name: 0, dtype: object

**Observation:** `iloc[0]` shows one customer as column-value pairs. So one row is one customer.

**ML relevance:** The model learns from many such records and then predicts for a new customer.

## 2. Dataset Dimensions

Dimensions mean the number of rows and columns. `df.shape` gives them as (rows, columns).

**Example:** 30 students and 4 columns gives shape (30, 4).  
**Why it matters:** The size tells us how much data we have for the model to learn from, and whether it is easy to handle on our system.

In [5]:
print("Shape:", df.shape)
print("Total cells:", df.size)
print("Memory (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))

Shape: (7043, 21)
Total cells: 147903
Memory (MB): 1.86


**Observation:** The data has 7043 rows and 21 columns and takes very little memory, so it is easy to work with on a normal laptop.

**ML relevance:** 7043 rows is enough data for normal ML models.

## 3. What is a Feature?

A feature is a column that describes something about each record. Features are the inputs that the ML model uses to learn.

**Example:** For a house, the features can be area, number of bedrooms and location.  
**In my dataset:** tenure, Contract, MonthlyCharges and so on.

In [6]:
for col in df.columns:
    print(col)

customerID
gender
SeniorCitizen
Partner
Dependents
tenure
PhoneService
MultipleLines
InternetService
OnlineSecurity
OnlineBackup
DeviceProtection
TechSupport
StreamingTV
StreamingMovies
Contract
PaperlessBilling
PaymentMethod
MonthlyCharges
TotalCharges
Churn


**Observation:** There are 21 columns. `customerID` is only an id and `Churn` is what we want to predict, so the remaining 19 are features. They describe the customer (gender, partner, dependents), the services taken (phone, internet, streaming) and the account (contract, payment, charges).

**ML relevance:** The model can only learn from the features we give it, so understanding them is important.

## 4. Target, Independent and Dependent Variables

- **Target variable:** the column we want to predict.
- **Dependent variable:** the value that depends on other variables. In ML this is the target (y).
- **Independent variables:** the inputs used to predict it (X).

**Example:** Marks depend on study hours. Marks is dependent, study hours is independent.  
**In my dataset:** Churn is the target (dependent). All other columns except customerID are independent variables.

In [7]:
y = df["Churn"]
X = df.drop(columns=["Churn", "customerID"])

print("X shape:", X.shape)
print("y shape:", y.shape)

print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

X shape: (7043, 19)
y shape: (7043,)
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


**Observation:** `Churn` has two values, No and Yes, so this is binary classification. Around 73.5% customers did not churn and around 26.5% churned. The classes are not equal, so the data is imbalanced.

**ML relevance:** With imbalanced data, accuracy alone can be misleading. A model that always says "No" would get about 73% accuracy but would never find a churner.

## 5. Numerical and Categorical Features

- **Numerical:** values are numbers and we can do maths on them (mean, sum). Example: age, salary.
- **Categorical:** values are labels or groups. Example: gender, city, payment method.

The data type shown by pandas may not match the real meaning of the column, so I check both.

In [8]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

**Observation:** Pandas shows only 3 numeric columns: `SeniorCitizen`, `tenure`, `MonthlyCharges`. But two columns look wrong:
- `SeniorCitizen` is 0/1, which is really Yes/No, so it is categorical.
- `TotalCharges` is shown as object (text), but it should be numerical.

I will check these two.

In [9]:
print(df["SeniorCitizen"].unique())

# try to convert TotalCharges to numbers, the ones that fail become NaN
total = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Cannot convert:", total.isna().sum())

df[total.isna()][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

[0 1]
Cannot convert: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


**Observation:** `SeniorCitizen` has only 0 and 1. In `TotalCharges` there are 11 rows that have a blank space instead of a number, and that is why pandas read the column as text. All these customers have `tenure` = 0, so they are new customers who have not been billed yet.

I am only noting this problem here. I will handle it in the preprocessing sprint.

In [10]:
numerical = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical = [c for c in df.columns if c not in numerical + ["customerID", "Churn"]]

print("Numerical:", numerical)
print("Categorical:", categorical)
print(len(numerical), len(categorical))

df[categorical].nunique()

Numerical: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
3 16


gender              2
SeniorCitizen       2
Partner             2
Dependents          2
PhoneService        2
MultipleLines       3
InternetService     3
OnlineSecurity      3
OnlineBackup        3
DeviceProtection    3
TechSupport         3
StreamingTV         3
StreamingMovies     3
Contract            3
PaperlessBilling    2
PaymentMethod       4
dtype: int64

**Observation:** 3 numerical and 16 categorical features. Most categorical columns have only 2-4 unique values.

**ML relevance:** The categorical columns will need encoding and the numerical columns may need scaling before the model.

## 6. Discrete and Continuous Features

- **Discrete:** countable whole numbers. Example: number of children.
- **Continuous:** can take any value in a range including decimals. Example: weight, price.

In [11]:
df2 = df.copy()
df2["TotalCharges"] = pd.to_numeric(df2["TotalCharges"], errors="coerce")

for col in numerical:
    print(col, "| unique:", df2[col].nunique(), "| min:", df2[col].min(), "| max:", df2[col].max())

tenure | unique: 73 | min: 0 | max: 72
MonthlyCharges | unique: 1585 | min: 18.25 | max: 118.75
TotalCharges | unique: 6530 | min: 18.8 | max: 8684.8


**Observation:** `tenure` goes from 0 to 72 in whole months with only 73 unique values, so it is discrete. `MonthlyCharges` and `TotalCharges` have decimals and thousands of unique values, so they are continuous.

**Why it matters:** The type decides which plot to use later, for example a histogram works well for continuous columns.

## 7. Identifier Columns

An identifier column gives a unique value to every record, like roll number or order id. It does not help in predicting anything.

**In my dataset:** `customerID`.

In [12]:
print("Rows:", len(df))
print("Unique ids:", df["customerID"].nunique())
print("Duplicate ids:", df["customerID"].duplicated().sum())

Rows: 7043
Unique ids: 7043
Duplicate ids: 0


**Observation:** The number of unique ids equals the number of rows, so each customer appears once.

**ML relevance:** Ids should be removed before training because the model could memorise them instead of learning real patterns. That is why I dropped it from X.

## 8. Metadata

Metadata means data about the data, like column names, data types, meaning and role of each column. I have made a small data dictionary for my dataset.

In [13]:
meta = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.values,
    "unique_values": df.nunique().values
})

def role(c):
    if c == "customerID":
        return "identifier"
    if c == "Churn":
        return "target"
    return "feature"

def kind(c):
    if c in ["customerID", "Churn"]:
        return "-"
    if c == "tenure":
        return "numerical (discrete)"
    if c in ["MonthlyCharges", "TotalCharges"]:
        return "numerical (continuous)"
    return "categorical"

meta["role"] = meta["column"].apply(role)
meta["feature_type"] = meta["column"].apply(kind)
meta

,column,dtype,unique_values,role,feature_type
0,customerID,str,7043,identifier,-
1,gender,str,2,feature,categorical
2,SeniorCitizen,int64,2,feature,categorical
3,Partner,str,2,feature,categorical
4,Dependents,str,2,feature,categorical
5,tenure,int64,73,feature,numerical (discrete)
6,PhoneService,str,2,feature,categorical
7,MultipleLines,str,3,feature,categorical
8,InternetService,str,3,feature,categorical
9,OnlineSecurity,str,3,feature,categorical


**Observation:** The table shows the dtype, role and real type of every column. Comparing dtype with feature_type shows the two mismatches: `SeniorCitizen` and `TotalCharges`.

This table will help me in the next notebooks and in the final report.

## Summary

- Telco Customer Churn dataset, 7043 rows and 21 columns.
- One row = one customer.
- Target is `Churn` (binary classification), around 73.5% No and 26.5% Yes, so it is imbalanced.
- `customerID` is the identifier and should not be used for modelling.
- Numerical: `tenure` (discrete), `MonthlyCharges`, `TotalCharges` (continuous). 16 categorical features.

**Problems found (to fix in preprocessing sprint):**
1. `TotalCharges` is stored as text and has 11 blank values.
2. `SeniorCitizen` is numeric but is actually categorical.
3. Target classes are imbalanced.
4. Categorical columns need encoding and numerical columns may need scaling.

**What I learned:** Before analysing data I should know what a row means, which column is the target, and what the real type of each column is, because the pandas dtype alone can be wrong.

Next: `02_Data_Inspection.ipynb`